> **🎯 Important**
>
**Durée estimée** : 8 à 10 heures  
**Prérequis** : toutes les notions du Module 5 (5.1 à 5.6) + Module 3  
**Objectif** : réaliser un projet de segmentation client complet avec visualisation, clustering et détection d'anomalies


# 🎯 Contexte

Tu es **data scientist** chez **ShopHub**, un e-commerce français en pleine croissance. L'équipe marketing te sollicite :

> « On a 5000 clients et on leur envoie **tous les mêmes emails marketing**. C'est clairement pas optimal.
> Est-ce que tu peux nous **segmenter intelligemment** les clients ? 
> On veut comprendre leurs **profils** et adapter nos campagnes. 
> Et tant qu'on y est : notre service fraude a repéré quelques **comptes suspects** — peux-tu identifier les **clients atypiques** ? »

## 💼 Les objectifs business

1. **Segmentation** : identifier des groupes de clients avec des **comportements similaires**
2. **Caractérisation** : décrire chaque segment (« qui sont-ils ? », « que font-ils ? »)
3. **Actions marketing** : proposer une stratégie différente pour chaque segment
4. **Détection d'anomalies** : identifier les clients au comportement **suspect** (fraudes, abus)

# 📊 Le dataset

Le fichier `clients_ecommerce.csv` contient 5000 clients avec 10 variables comportementales :

| Colonne | Description |
|---|---|
| `id_client` | Identifiant unique |
| `anciennete_jours` | Ancienneté du compte (jours depuis l'inscription) |
| `nb_commandes_total` | Nombre total de commandes passées |
| `montant_total_eur` | Montant total dépensé (€) |
| `panier_moyen_eur` | Montant moyen par commande (€) |
| `jours_depuis_derniere_cmd` | Jours depuis la dernière commande |
| `nb_categories_achetees` | Nombre de catégories de produits différentes |
| `taux_retour_pct` | % des commandes retournées |
| `nb_avis_laisses` | Nombre d'avis clients laissés |
| `note_moyenne_donnee` | Note moyenne donnée par le client (0-5) |

> **⚠️ Attention**
>
## ⚠️ Particularités du dataset
- Quelques **valeurs manquantes** sur `taux_retour_pct` et `note_moyenne_donnee`
- Certains clients ont un comportement **très atypique** (anomalies à découvrir)
- Les **échelles des variables diffèrent énormément** (jours vs euros vs pourcentages)


# 🗺️ Plan du TP

Le TP se structure en **6 parties** qui suivent le workflow d'un projet de segmentation professionnelle :

1. **Exploration** — comprendre le dataset
2. **Preprocessing** — nettoyer et préparer
3. **Visualisation** — voir la structure des données (PCA, UMAP)
4. **Clustering** — identifier les segments
5. **Caractérisation** — nommer et décrire chaque segment
6. **Détection d'anomalies** — repérer les comportements suspects

# 🛠️ Mise en route

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest

import umap  # pip install umap-learn

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 20)
np.random.seed(42)

print("✅ Environnement prêt")

## 📥 Préparation : téléchargement des données

La cellule ci-dessous télécharge automatiquement les datasets nécessaires si ils ne sont pas déjà présents localement. Cela permet de faire marcher le notebook **aussi bien en local qu'en Google Colab**.

In [ ]:
# 📥 Téléchargement automatique des datasets (utile pour Colab)
import os, urllib.request

if not os.path.exists('ressources_tp/clients_ecommerce.csv'):
    os.makedirs('ressources_tp', exist_ok=True)
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/Digit-Factory/python-ia-formation/main/ressources_eleves/module_05/ressources_tp/clients_ecommerce.csv',
        'ressources_tp/clients_ecommerce.csv'
    )
    print(f"✅ Dataset téléchargé : clients_ecommerce.csv")
else:
    print(f"✅ Dataset déjà présent : clients_ecommerce.csv")

In [ ]:
df = pd.read_csv("ressources_tp/clients_ecommerce.csv")

In [ ]:
# Charger le dataset
df = pd.read_csv("ressources_tp/clients_ecommerce.csv")
print(f"Dataset chargé : {df.shape}")

---

# Partie 1 — Exploration

## ✏️ Étape 1.1 — Premier coup d'œil

> **ℹ️ Note**
>
## 📝 À faire

1. Affiche `df.shape` et `df.head()`
2. Affiche `df.info()` et `df.describe()`
3. Repère les **points marquants** : quelle(s) colonne(s) ont des valeurs manquantes ? Les échelles sont-elles homogènes ?

In [ ]:
# TODO: Étape 1.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.2 — Explorer les distributions

> **ℹ️ Note**
>
## 📝 À faire

Trace un **histogramme** pour chacune des 9 features numériques (pas `id_client`).

Observations attendues :
- Y a-t-il des distributions **bimodales** (plusieurs pics) ?
- Des **outliers extrêmes** visibles ?
- Des distributions **asymétriques** (queue longue) ?

In [ ]:
# TODO: Étape 1.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 1.3 — Corrélations

> **ℹ️ Note**
>
## 📝 À faire

Trace la **matrice de corrélation** des 9 variables numériques (heatmap).

1. Quelles variables sont fortement corrélées ?
2. Cela donne-t-il des indices sur les profils possibles ?

In [ ]:
# TODO: Étape 1.3

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 2 — Preprocessing

## ✏️ Étape 2.1 — Gérer les valeurs manquantes

> **ℹ️ Note**
>
## 📝 À faire

1. Mets de côté `id_client` (pas utilisable pour le clustering)
2. Impute les valeurs manquantes avec la **médiane** via `SimpleImputer`
3. Vérifie qu'il ne reste plus aucun NaN

In [ ]:
# TODO: Étape 2.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 2.2 — Standardiser

> **ℹ️ Note**
>
## 📝 À faire

1. Applique `StandardScaler` sur `X_imp`
2. Vérifie que la **moyenne est proche de 0** et l'**écart-type de 1** pour chaque feature

In [ ]:
# TODO: Étape 2.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 3 — Visualisation

## ✏️ Étape 3.1 — PCA : combien de vraies dimensions ?

> **ℹ️ Note**
>
## 📝 À faire

1. Applique une PCA complète (toutes composantes)
2. Trace la **variance cumulée** expliquée
3. Combien de composantes pour atteindre **90% de variance** ?

In [ ]:
# TODO: Étape 3.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 3.2 — Visualiser en 2D avec UMAP

> **ℹ️ Note**
>
## 📝 À faire

1. Applique UMAP (`n_neighbors=30`, `min_dist=0.1`) pour obtenir une projection 2D
2. Affiche les 5000 clients en scatter plot
3. Vois-tu **des groupes** à l'œil nu ?

In [ ]:
# TODO: Étape 3.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 4 — Clustering

## ✏️ Étape 4.1 — Choisir k avec la méthode du coude et la silhouette

> **ℹ️ Note**
>
## 📝 À faire

Pour `k` de 2 à 10 :
1. Applique `KMeans(n_clusters=k, n_init=10, random_state=42)` sur `X_scaled`
2. Stocke l'**inertie** et le **score de silhouette**
3. Trace les deux courbes
4. Quel k te semble optimal ?

In [ ]:
# TODO: Étape 4.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 4.2 — Appliquer k-means avec k choisi

> **ℹ️ Note**
>
## 📝 À faire

1. Entraîne `KMeans(n_clusters=5, n_init=10, random_state=42)` sur `X_scaled`
2. Ajoute les labels de cluster comme nouvelle colonne du DataFrame original
3. Affiche la **taille de chaque cluster**
4. Visualise les clusters sur la projection UMAP

In [ ]:
# TODO: Étape 4.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 5 — Caractérisation et recommandations

## ✏️ Étape 5.1 — Profil moyen de chaque cluster

> **ℹ️ Note**
>
## 📝 À faire

1. Calcule la **moyenne de chaque feature par cluster**
2. Range le résultat dans un DataFrame pour comparaison facile
3. Identifie ce qui distingue chaque cluster

In [ ]:
# TODO: Étape 5.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 5.2 — Nommer les segments et proposer des actions

> **ℹ️ Note**
>
## 📝 À faire

En te basant sur les profils :

1. **Donne un nom métier** à chaque cluster (ex: "Champions", "Dormants"...)
2. Pour chaque segment, propose **une action marketing concrète**

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# Partie 6 — Détection d'anomalies

## ✏️ Étape 6.1 — Isolation Forest

> **ℹ️ Note**
>
## 📝 À faire

L'équipe fraude soupçonne **environ 1%** de clients atypiques (fraude, abus, comptes tests).

1. Applique un `IsolationForest(contamination=0.01, random_state=42)` sur `X_scaled`
2. Ajoute une colonne `anomalie` (1 = anomalie, 0 = normal) au DataFrame
3. Combien de clients détectés ?
4. Visualise-les sur la projection UMAP

In [ ]:
# TODO: Étape 6.1

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 6.2 — Analyser les anomalies détectées

> **ℹ️ Note**
>
## 📝 À faire

1. Extrait les clients flaggés comme anomalies
2. Compare leur **profil moyen** avec celui des clients normaux
3. Peux-tu **hypothèses** sur le type d'anomalie (fraude ? abus ? compte test ?)

In [ ]:
# TODO: Étape 6.2

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

## ✏️ Étape 6.3 — Anomalies par cluster ?

> **ℹ️ Note**
>
## 📝 À faire

1. Crée un **tableau croisé** `cluster × anomalie`
2. Dans quel(s) cluster(s) se concentrent les anomalies ?

> 💡 **Tu bloques ?** Consulte la correction sur le site web du cours en dépliant le bloc « Voir la correction ».

---

# 🎓 Bilan du TP

## 🏆 Ce que tu viens d'accomplir

Tu as mené un **projet ML non-supervisé complet** :

- ✅ Exploration et nettoyage (Module 3 + 5)
- ✅ Preprocessing industriel (imputation + standardisation)
- ✅ Visualisation haute dimension via UMAP
- ✅ Sélection de `k` avec 2 méthodes (coude + silhouette)
- ✅ Clustering k-means avec 5 segments
- ✅ Caractérisation métier et actions recommandées
- ✅ Détection d'anomalies avec Isolation Forest

C'est **exactement** le workflow d'un projet de segmentation en entreprise.

## 🔍 Les notions mobilisées

| Notion | Usage |
|---|---|
| **5.1 — Intro** | Cadre général, tableau comparatif |
| **5.2 — k-means** | Clustering principal, silhouette |
| **5.4 — PCA** | Analyse de la variance, base de l'intuition |
| **5.5 — UMAP** | Visualisation des segments |
| **5.6 — Anomalies** | Détection des clients atypiques |
| **Module 3** | Preprocessing, imputation, standardisation |

## 💡 Pistes pour aller plus loin

1. **Tester d'autres algorithmes** : DBSCAN, clustering hiérarchique (Ward)
2. **Analyser chaque cluster avec UMAP séparé** pour voir les sous-groupes
3. **Créer un dashboard interactif** (Plotly/Dash) pour l'équipe marketing
4. **Mettre en production** : pipeline scikit-learn complet, sauvegarde via `joblib`
5. **Monitoring** : re-entraîner le modèle tous les mois pour détecter le **drift**

## 📊 Le rapport exécutif (bonus)

> **💡 Astuce**
>
## 📝 Exemple de résumé à présenter

**Rapport de segmentation clients — ShopHub**

*Méthodologie* : analyse non-supervisée de 5000 clients sur 9 indicateurs comportementaux.

*Résultats* :
- **5 segments naturels** identifiés (silhouette 0.XX)
- **Champions** (12%) : 80% du chiffre d'affaires → focus programme VIP
- **Fidèles** (30%) : socle stable → personnalisation
- **Prometteurs** (18%) : upsell prioritaire
- **Dormants** (26%) : campagne de réactivation massive
- **À risque** (13%) : enquête satisfaction urgente

*Alertes* :
- **50 clients flaggés** comme atypiques
- 20 cas probables de fraude (comptes récents, gros volumes)
- 20 cas d'abus (taux de retour > 70%)
- 10 comptes tests internes à nettoyer

*Prochaines étapes* :
1. Validation métier des segments avec l'équipe marketing
2. Lancement des campagnes différenciées (semaine 1-2)
3. Enquête sur les 20 fraudes probables (équipe risque)
4. Re-segmentation trimestrielle


**🎉 Bravo — tu as bouclé le Module 5 !**

## 🚀 La suite

Tu as maintenant :
- ✅ Les fondations (Modules 1-3)
- ✅ Le ML supervisé (Module 4)
- ✅ Le ML non-supervisé (Module 5)

Les prochains modules vont te faire découvrir :

- **Module 6 — Deep Learning** : réseaux de neurones, CNN, PyTorch
- **Module 7 — LLM et IA générative** : embeddings, RAG, agents
- **Module 8 — MLOps** : déploiement, monitoring, MLflow

Tu as déjà largement de quoi **être autonome** sur des projets ML classiques en entreprise !